<a href="https://colab.research.google.com/github/Prashkov1ch/python-ai-Prashkovich-Anna/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- [`seas-square-ocean-named after.csv`](https://github.com/Prashkov1ch/python-ai-Prashkovich-Anna/blob/main/data/seas-square-ocean-named%20after.csv) — информация о морях:
  - `sea` — Wikidata ID сущности
  - `seaLabel` — название моря/океана на русском
  - `oceanLabel` — принадлежность к океану
  - `area` — площадь (км²)
  - `depth` — глубина (м)
  - `coordinates` — гео-координаты в формате WKT
  - `named_afterLabel` — объект, в честь которого названо

**Что мы делаем:**
1. Клонируем ваш репозиторий GitHub в Colab
2. Читаем CSV-файл с данными об океанах в pandas DataFrame
3. Очищаем и переименовываем столбцы при необходимости
4. Смотрим структуру данных, типы и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [2]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

repo = "python-ai-Prashkovich-Anna"  # 🔧 ИЗМЕНЕНО: ваше имя репозитория
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Prashkovich-Anna
    !git clone -q https://github.com/Prashkov1ch/python-ai-Prashkovich-Anna.git  # 🔧 ИЗМЕНЕНО: ваш URL

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

/content/python-ai-Prashkovich-Anna
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Prashkovich-Anna


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [3]:
# 🐱 Шаг 2A. Чтение CSV-файла в pandas

import pandas as pd

# 🔧 ИЗМЕНЕНО: читаем ваш файл с данными об океанах
# Обратите внимание: в имени файла есть пробел — pandas справится, но будьте внимательны
df_oceans = pd.read_csv("data/seas-square-ocean-named after.csv")

print("✅ Загружено строк в df_oceans:", len(df_oceans))
print("✅ Колонки:", df_oceans.columns.tolist())

✅ Загружено строк в df_oceans: 214
✅ Колонки: ['sea', 'seaLabel', 'oceanLabel', 'area', 'depth', 'coordinates', 'named_afterLabel']


## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `sea` с URL (ссылкой на объект Wikidata) — **удаляем**, так как для базового анализа он не нужен.
- Столбцы `seaLabel`, `oceanLabel`, `named_afterLabel` содержат читаемые подписи (названия) — **переименуем**, убирая суффикс `Label`.

В этом шаге мы:
- удалим столбец с URL Wikidata (`sea`);
- переименуем `seaLabel → sea`, `oceanLabel → ocean`, `named_afterLabel → named_after`;
- приведём числовые столбцы (`area`, `depth`) к типу `int`/`float` для корректных вычислений.

При приведении к числам мы используем:

- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(int)` или `astype(float)` — переводит столбец к нужному числовому типу.

> ⚠️ **Важно:** если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа. Столбец с URL можно сохранить, если позже понадобится переходить к оригинальным записям в Викиданных.

In [4]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для датасета об океанах

# 🔧 ИЗМЕНЕНО: работаем с вашим DataFrame df_oceans

# 1) Удаляем технический столбец с URL Wikidata (если существует)
if "sea" in df_oceans.columns:
    df_oceans = df_oceans.drop(columns=["sea"], errors="ignore")  # 🔧 ИЗМЕНЕНО: удаляем sea с URL
    print("🗑️ Столбец 'sea' (URL) удалён")

# 2) Переименовываем столбцы: убираем суффикс Label
if "seaLabel" in df_oceans.columns:  # Проверка: нужна ли очистка?
    df_oceans = df_oceans.rename(columns={
        "seaLabel": "sea",                # 🔧 ИЗМЕНЕНО: ваше переименование
        "oceanLabel": "ocean",            # 🔧 ИЗМЕНЕНО: ваше переименование
        "named_afterLabel": "named_after" # 🔧 ИЗМЕНЕНО: ваше переименование
    })
    print("✅ Столбцы переименованы: *Label → без суффикса")

# 3) Приводим числовые столбцы к правильным типам
# area — площадь в км² (целое число), depth — глубина в м (может быть дробной)
if "area" in df_oceans.columns:
    df_oceans["area"] = pd.to_numeric(
        df_oceans["area"], errors="coerce"
    ).fillna(0).astype(int)  # 🔧 ИЗМЕНЕНО: площадь как целое число

if "depth" in df_oceans.columns:
    df_oceans["depth"] = pd.to_numeric(
        df_oceans["depth"], errors="coerce"
    ).fillna(0).astype(float)  # 🔧 ИЗМЕНЕНО: глубина как float (точность важна)

print("\n✅ Данные об океанах готовы к анализу")
print("📋 Итоговые колонки:", df_oceans.columns.tolist())

🗑️ Столбец 'sea' (URL) удалён
✅ Столбцы переименованы: *Label → без суффикса

✅ Данные об океанах готовы к анализу
📋 Итоговые колонки: ['sea', 'ocean', 'area', 'depth', 'coordinates', 'named_after']


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame с данными об океанах:

- посмотрим размер таблицы (количество строк и столбцов);
- выведем список всех столбцов;
- посмотрим первые несколько строк для понимания структуры;
- посчитаем базовую статистику по числовым полям (`area`, `depth`).

**Зачем это нужно:**
- Убедиться, что данные загрузились корректно.
- Понять, какие столбцы требуют очистки или переименования.
- Оценить диапазоны числовых значений (площадь, глубина).
- Выявить очевидные проблемы (пропуски, странные форматы).

**Что мы увидим:**
| Показатель | Описание |
|------------|----------|
| `shape` | Количество строк и столбцов в таблице |
| `columns` | Список всех имён столбцов |
| `head()` | Первые 5 строк для быстрого просмотра |
| `describe()` | Статистика по числовым колонкам (min, max, mean, std) |

> **Важно:** Этот шаг выполняется **до очистки данных**, чтобы увидеть исходное состояние и потом сравнить с результатом после Шага 2B.

In [5]:
# 🔍 Шаг 3. Обзор данных: структура и первые строки

def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n{'='*60}")
    print(f"📊 {name}")
    print('='*60)
    print("📏 Размер таблицы (строки, столбцы):", df.shape)
    print("📋 Столбцы:", ", ".join(df.columns))
    print(f"\n🔢 Первые {n} строк:")
    display(df.head(n))

    # Проверка на пропуски
    null_counts = df.isnull().sum()
    if null_counts.any():
        print("\n⚠️ Пропуски в столбцах:")
        for col, count in null_counts[null_counts > 0].items():
            print(f"   {col}: {count} пропусков ({count/len(df)*100:.1f}%)")
    else:
        print("\n✅ Пропусков не обнаружено")

# 🚀 Запуск обзора для основного DataFrame
show_info(df_oceans, "Моря и океаны (df_oceans)", n=5)

# 📈 Бонус: базовая статистика по числовым колонкам
print("\n" + "="*60)
print("📈 Базовая статистика по числовым столбцам")
print("="*60)

numeric_cols = df_oceans.select_dtypes(include=['number']).columns
if len(numeric_cols) > 0:
    print(f"Числовые столбцы: {', '.join(numeric_cols)}\n")
    print(df_oceans[numeric_cols].describe().round(2))
else:
    print("⚠️ Числовые столбцы не найдены")

# 📋 Дополнительно: уникальные значения в текстовых колонках
print("\n" + "="*60)
print("🏷️ Уникальные значения в ключевых столбцах")
print("="*60)

text_cols_to_check = ['oceanLabel', 'named_afterLabel']  # до очистки
for col in text_cols_to_check:
    if col in df_oceans.columns:
        unique_count = df_oceans[col].nunique()
        print(f"\n{col}: {unique_count} уникальных значений")
        print("Топ-5 наиболее частых:")
        print(df_oceans[col].value_counts().head(5).to_string())

print("\n" + "="*60)
print("✅ Обзор данных завершён. Переходим к очистке (Шаг 2B).")
print("="*60)


📊 Моря и океаны (df_oceans)
📏 Размер таблицы (строки, столбцы): (214, 6)
📋 Столбцы: sea, ocean, area, depth, coordinates, named_after

🔢 Первые 5 строк:


,sea,ocean,area,depth,coordinates,named_after
0,Филиппинское море,Тихий океан,5000000,10911.0,Point(134.0 18.0),NaN
1,Аравийское море,Индийский океан,3862000,5800.0,Point(64.0 16.0),NaN
2,Южно-Китайское море,Тихий океан,3500000,5559.0,Point(113.0 12.0),NaN
3,Южно-Китайское море,Китайские моря,3500000,5559.0,Point(113.0 12.0),NaN
4,Карибское море,Северная Атлантика,2754000,7686.0,Point(-75.0 15.0),NaN



⚠️ Пропуски в столбцах:
   ocean: 64 пропусков (29.9%)
   coordinates: 40 пропусков (18.7%)
   named_after: 153 пропусков (71.5%)

📈 Базовая статистика по числовым столбцам
Числовые столбцы: area, depth

             area     depth
count      214.00    214.00
mean    243586.03    880.60
std     676141.41   1954.85
min          0.00      0.00
25%          0.00      0.00
50%          0.00      0.00
75%      43150.00    210.75
max    5000000.00  10911.00

🏷️ Уникальные значения в ключевых столбцах

✅ Обзор данных завершён. Переходим к очистке (Шаг 2B).


## 🔤 [4] Лингвистический детектив: кто «даёт имена» морям? (табличный анализ)

**Вопрос исследования**: В честь чего или кого чаще всего называют моря?

**Что мы считаем (без визуализации)**:
1. Топ-N самых частых значений в `named_after` — таблица с рангом, именем и количеством.
2. Автоматическую категоризацию значений по эвристикам:
   - 🌍 Страны/регионы
   - 👤 Люди (исследователи, правители)
   - ⚡ Мифология (боги, персонажи)
   - 🗻 Гео-объекты и направления
   - ❓ Другое / не определено
3. Сводную таблицу: сколько морей в каждой категории + процент от общего числа.
4. Кросс-таблицу: какая категория преобладает в каждом океане.

**Формат вывода**:
- Все результаты — в виде pandas DataFrame с `to_string()` или `to_markdown()`.
- Текстовые выводы с интерпретацией: «Наиболее частый источник названий — ...».
- Готовность к экспорту: таблицы можно скопировать или сохранить в CSV.

In [6]:
# 🔤 Шаг 4. Лингвистический анализ: кто «даёт имена» морям? (ТАБЛИЦЫ + ТЕКСТ)

import pandas as pd
import re

print("🔤 ЛИНГВИСТИЧЕСКИЙ АНАЛИЗ: столбец 'named_after'")
print("="*60)

# 1) Проверка наличия данных
if "named_after" not in df_oceans.columns or df_oceans["named_after"].isna().all():
    print("⚠️ Столбец 'named_after' пуст или отсутствует — пропускаем анализ")
else:
    # Очистка данных
    named_data = df_oceans["named_after"].dropna().astype(str).str.strip()
    named_data = named_data[named_data != "nan"]

    print(f"✅ Записей с named_after: {len(named_data)} из {len(df_oceans)}")
    print(f"🔢 Уникальных значений: {named_data.nunique()}\n")

    # 2) Топ-15 источников названий (таблица)
    print("📋 1. Топ-15 объектов, в честь которых названы моря:")
    print("-"*60)
    top_named = named_data.value_counts().head(15).reset_index()
    top_named.columns = ["Объект (named_after)", "Количество морей"]
    top_named["Ранг"] = range(1, len(top_named)+1)
    top_named = top_named[["Ранг", "Объект (named_after)", "Количество морей"]]
    print(top_named.to_string(index=False))
    print()

    # 3) Автоматическая категоризация (эвристики)
    def categorize_named(name):
        name_lower = str(name).lower()

        # Мифология
        myth_keywords = ["бог", "богиня", "посейдон", "нептун", "атлант", "титан", "миф", "олимп", "зеус"]
        if any(k in name_lower for k in myth_keywords):
            return "⚡ Мифология"

        # Люди: суффиксы, паттерны имён
        if re.search(r"[а-яё]+[а-яё]*(ов|ев|ин|ын|ский|ской|вич|на|ич)\b", name_lower):
            return "👤 Люди"
        if re.search(r"^[А-Я][а-яё]+\s+[А-Я][а-яё]+", str(name)):
            return "👤 Люди"

        # Страны/регионы
        country_keywords = ["россия", "китай", "индия", "япония", "норвегия", "швеция",
                           "германия", "франция", "испания", "италия", "америка", "европа",
                           "азия", "африка", "арктика", "антарктика", "латин", "рим"]
        if any(k in name_lower for k in country_keywords):
            return "🌍 Страны/регионы"

        # Гео-объекты и направления
        geo_keywords = ["север", "юг", "запад", "восток", "красн", "бел", "чёрн", "жёлт",
                       "гор", "мор", "рек", "остров", "пролив", "берег"]
        if any(k in name_lower for k in geo_keywords):
            return "🗻 Гео-объекты"

        return "❓ Другое"

    # Применяем категоризацию
    df_oceans["named_category"] = named_data.apply(categorize_named)

    # 4) Сводная таблица по категориям
    print("📋 2. Распределение по категориям источников названий:")
    print("-"*60)
    category_stats = df_oceans["named_category"].value_counts().reset_index()
    category_stats.columns = ["Категория", "Количество"]
    category_stats["Процент"] = (category_stats["Количество"] / len(df_oceans) * 100).round(1).astype(str) + "%"
    category_stats = category_stats[["Категория", "Количество", "Процент"]]
    print(category_stats.to_string(index=False))
    print()

    # Текстовый вывод-инсайт
    top_category = category_stats.iloc[0]
    print(f"💡 Инсайт: Чаще всего моря называют в честь '{top_category['Категория']}' "
          f"({top_category['Процент']} от всех записей)")
    print()

    # 5) Кросс-анализ: категории по океанам (сводная таблица)
    if "ocean" in df_oceans.columns and df_oceans["ocean"].notna().any():
        print("📋 3. Преобладающая категория named_after по океанам:")
        print("-"*60)

        cross_table_list = []
        for ocean_name in sorted(df_oceans["ocean"].dropna().unique()):
            subset = df_oceans[df_oceans["ocean"] == ocean_name]
            if len(subset) > 0:
                cat_counts = subset["named_category"].value_counts()
                if not cat_counts.empty:
                    top_cat = cat_counts.idxmax()
                    top_count = cat_counts.max()
                    cross_table_list.append({
                        "Океан": ocean_name,
                        "Всего морей": len(subset),
                        "Топ-категория": top_cat,
                        "В категории": top_count,
                        "Доля": f"{top_count/len(subset)*100:.1f}%"
                    })

        if cross_table_list:
            cross_df = pd.DataFrame(cross_table_list)
            print(cross_df.to_string(index=False))
        else:
            print("⚠️ Недостаточно данных для кросс-анализа")

    # 6) Бонус: детализация по категории "Люди" (если есть)
    if "👤 Люди" in df_oceans["named_category"].values:
        print("\n📋 4. Детализация: моря, названные в честь людей:")
        people_seas = df_oceans[df_oceans["named_category"] == "👤 Люди"][["sea", "named_after", "ocean"]]
        if len(people_seas) > 0:
            print(people_seas.to_string(index=False))
        else:
            print("⚠️ Нет записей в категории 'Люди'")

print("\n" + "="*60)
print("✅ Анализ завершён. Все результаты — в табличном формате.")

🔤 ЛИНГВИСТИЧЕСКИЙ АНАЛИЗ: столбец 'named_after'
✅ Записей с named_after: 61 из 214
🔢 Уникальных значений: 45

📋 1. Топ-15 объектов, в честь которых названы моря:
------------------------------------------------------------
 Ранг    Объект (named_after)  Количество морей
    1                   север                 5
    2         Индийский океан                 4
    3             Мигель Грау                 2
    4                  Scotia                 2
    5           Уильям Баффин                 2
    6                  жёлтый                 2
    7          Фрэнсис Бофорт                 2
    8                   Адрия                 2
    9                      юг                 2
   10                 Касситы                 2
   11 Михаил Петрович Лазарев                 2
   12   Витус Ионассен Беринг                 1
   13 Михаил Михайлович Сомов                 1
   14                 красный                 1
   15                германцы                 1

📋 2. Рас

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨